In [38]:
import pandas as pd
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory

In [39]:
lucro = pd.read_excel('exerc4.xlsx',sheet_name='lucro', index_col=0).T.to_dict()
demanda = pd.read_excel('exerc4.xlsx',sheet_name='demanda', index_col=0)['demanda'].to_dict()
dict_producao = pd.read_excel('exerc4.xlsx',sheet_name='producao', index_col=0)['producao'].to_dict()

In [40]:
lucro

{'f1': {'fregues1': 65, 'fregues2': 63, 'fregues3': 62, 'fregues4': 64},
 'f2': {'fregues1': 68, 'fregues2': 67, 'fregues3': 65, 'fregues4': 62},
 'f3': {'fregues1': 63, 'fregues2': 60, 'fregues3': 59, 'fregues4': 60}}

In [41]:
demanda

{'fregues1': 4000, 'fregues2': 3000, 'fregues3': 1000, 'fregues4': 0}

In [42]:
dict_producao

{'f1': 3000, 'f2': 5000, 'f3': 4000}

In [43]:
model = pyo.ConcreteModel()

model.fabricas = pyo.Set(initialize=dict_producao.keys())
model.fregueses = pyo.Set(initialize=demanda.keys())
model.x = pyo.Var(model.fabricas, model.fregueses, bounds=(0,None), domain=NonNegativeIntegers)

def objetivo(model):
    return sum(
        model.x[fa,fr] * lucro[fa][fr] for fa in model.fabricas for fr in model.fregueses
    )
model.obj = pyo.Objective(rule=objetivo, sense=pyo.maximize)

def restricao_demanda(model, fr):
    return sum(
        model.x[fa,fr] for fa in model.fabricas
    ) >= demanda[fr]
model.restricao_demand_a = pyo.Constraint(model.fregueses, rule=restricao_demanda)

def producao(model, fa):
    return sum(
        model.x[fa,fr] for fr in model.fregueses
    ) == dict_producao[fa]
model.producao = pyo.Constraint(model.fabricas, rule=producao)

In [44]:
model.pprint()

2 Set Declarations
    fabricas : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    3 : {'f1', 'f2', 'f3'}
    fregueses : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    4 : {'fregues1', 'fregues2', 'fregues3', 'fregues4'}

1 Var Declarations
    x : Size=12, Index=fabricas*fregueses
        Key                : Lower : Value : Upper : Fixed : Stale : Domain
        ('f1', 'fregues1') :     0 :  None :  None : False :  True : NonNegativeIntegers
        ('f1', 'fregues2') :     0 :  None :  None : False :  True : NonNegativeIntegers
        ('f1', 'fregues3') :     0 :  None :  None : False :  True : NonNegativeIntegers
        ('f1', 'fregues4') :     0 :  None :  None : False :  True : NonNegativeIntegers
        ('f2', 'fregues1') :     0 :  None :  None : False :  True : NonNegativeIntegers
        ('f2', 'fregues2') :     0 :  None :  None 

In [45]:
opt = SolverFactory('cplex')
resultado = opt.solve(model, tee=True)


Welcome to IBM(R) ILOG(R) CPLEX(R) Interactive Optimizer 22.1.1.0
  with Simplex, Mixed Integer & Barrier Optimizers
5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55 5655-Y21
Copyright IBM Corp. 1988, 2022.  All Rights Reserved.

Type 'help' for a list of available commands.
Type 'help' followed by a command name for more
information on commands.

CPLEX> Logfile 'cplex.log' closed.
Logfile 'C:\Users\DECIV\AppData\Local\Temp\tmptpq334is.cplex.log' open.
CPLEX> Problem 'C:\Users\DECIV\AppData\Local\Temp\tmpnbphqu5c.pyomo.lp' read.
Read time = 0.00 sec. (0.00 ticks)
CPLEX> Problem name         : C:\Users\DECIV\AppData\Local\Temp\tmpnbphqu5c.pyomo.lp
Objective sense      : Maximize
Variables            :      12  [General Integer: 12]
Objective nonzeros   :      12
Linear constraints   :       7  [Greater: 4,  Equal: 3]
  Nonzeros           :      24
  RHS nonzeros       :       6

Variables            : Min LB: 0.000000         Max UB: all infinite   
Objective nonzeros   : Min   : 

In [46]:
model.x.pprint()

x : Size=12, Index=fabricas*fregueses
    Key                : Lower : Value  : Upper : Fixed : Stale : Domain
    ('f1', 'fregues1') :     0 : 3000.0 :  None : False : False : NonNegativeIntegers
    ('f1', 'fregues2') :     0 :   -0.0 :  None : False : False : NonNegativeIntegers
    ('f1', 'fregues3') :     0 :   -0.0 :  None : False : False : NonNegativeIntegers
    ('f1', 'fregues4') :     0 :   -0.0 :  None : False : False : NonNegativeIntegers
    ('f2', 'fregues1') :     0 : 1000.0 :  None : False : False : NonNegativeIntegers
    ('f2', 'fregues2') :     0 : 3000.0 :  None : False : False : NonNegativeIntegers
    ('f2', 'fregues3') :     0 : 1000.0 :  None : False : False : NonNegativeIntegers
    ('f2', 'fregues4') :     0 :   -0.0 :  None : False : False : NonNegativeIntegers
    ('f3', 'fregues1') :     0 : 4000.0 :  None : False : False : NonNegativeIntegers
    ('f3', 'fregues2') :     0 :   -0.0 :  None : False : False : NonNegativeIntegers
    ('f3', 'fregues3') :     

In [54]:
lista = []
dict_final = {}
for fa in model.fabricas:
    for fr in model.fregueses:
        dict_final = {
            fa: {
                fr: pyo.value(model.x[fa,fr])
            }
        }
        lista.append(dict_final)


In [93]:
for a,b in model.x:
    if b == 'fregues1':
        print(pyo.value(model.x[a,b]))

3000.0
1000.0
4000.0


In [47]:
pyo.value(model.obj)

781000.0